In [ ]:
from pathlib import Path
import xarray as xr
import torch
import torch.nn.functional
from climanet.st_encoder_decoder import SpatioTemporalModel
from climanet.utils import set_seed, configure_compute_resources, plot_results, plot_histograms, plot_loss, data_preparation, read_st_data
from climanet.train import train_monthly_model, TrainConfig
from climanet.predict import predict_monthly_var, PredictionConfig
from climanet.dataset import STDataset, DataLoaderConfig

from tbparse import SummaryReader
import matplotlib.pyplot as plt

## Read data

In [ ]:
# without diurnal cycle
data_folder = Path("/home/sarah/temp/eso4clima/dc_data")
run_dir = "./runs_daily"

var_name = "tos"

# 1 month train, 1 month validation and test
daily_data = xr.open_mfdataset(data_folder / f"202101_day_ERA5dc_masked_{var_name}.nc")
daily_data_validation = xr.open_mfdataset(data_folder / f"202102_day_ERA5dc_masked_{var_name}.nc")
daily_data_test = xr.open_mfdataset(data_folder / f"202103_day_ERA5dc_masked_{var_name}.nc")

monthly_data = xr.open_mfdataset(data_folder / f"202101_mon_ERA5dc_full_{var_name}.nc")
monthly_data_validation = xr.open_mfdataset(data_folder / f"202102_mon_ERA5dc_full_{var_name}.nc")
monthly_data_test = xr.open_mfdataset(data_folder / f"202103_mon_ERA5dc_full_{var_name}.nc")

file_name = data_folder / "era5_lsm_bool.nc"  # downloded from era5 and regridded using the function `regrid_to_boundary_centered_grid`
lsm_mask = xr.open_dataset(file_name)
# lsm_mask = lsm_mask.rename({'latitude': 'lat', 'longitude': 'lon'})

## Subset data (for fast example)

In [ ]:
# coordinates of subset
lon_subset = slice(-50, 50)  # one lon -179.9 is nan, check data
lat_subset = slice(-30, 10)

daily_subset = daily_data.sel(lon=lon_subset, lat=lat_subset)
monthly_subset = monthly_data.sel(lon=lon_subset, lat=lat_subset)
lsm_subset = lsm_mask.sel(lon=lon_subset, lat=lat_subset)  # True=Land

daily_validation_subset = daily_data_validation.sel(lon=lon_subset, lat=lat_subset)
monthly_validation_subset = monthly_data_validation.sel(lon=lon_subset, lat=lat_subset)

daily_test_subset = daily_data_test.sel(lon=lon_subset, lat=lat_subset)
monthly_test_subset = monthly_data_test.sel(lon=lon_subset, lat=lat_subset)

print(daily_subset[var_name].shape, monthly_subset[var_name].shape)  # (time, lat, lon)

## Training workflow

### Build the model

In [ ]:
# create the model (small)
set_seed()

In [ ]:
patch_size = (1, 4, 4)
model = SpatioTemporalModel(patch_size=patch_size, overlap=2, embed_dim=64, dropout=0.2, hidden=64)

In [ ]:
# Device and resources
device = "cpu"
compute_threads = 4
dataloader_num_workers = 1
model = configure_compute_resources(model, device=device, compute_threads=compute_threads, dataloader_num_workers=dataloader_num_workers)

### Build the datasets

#### Prepare the train data

In [ ]:
data_dir = f"{run_dir}/data_train"
input_da, input_da_nan_mask, monthly_da, padded_days_mask, time_features = data_preparation(
    daily_subset[var_name], monthly_subset[var_name], calculate_residuals=True, is_hourly=False, save_to_zarr=True, run_dir=data_dir,
)

In [ ]:
# read data 
data_dir = f"{run_dir}/data_train"
input_da, input_da_nan_mask, monthly_da, padded_days_mask, time_features = read_st_data(data_path=data_dir, var_name=var_name)

In [ ]:
# create dataset config
num_patches = (10, 10)
spatial_patch_size = (patch_size[1]*num_patches[0], patch_size[2]*num_patches[1])
stride = (spatial_patch_size[0] // 5, spatial_patch_size[1] // 5)

dataset_train = STDataset(
    input_da=input_da,
    input_da_nan_mask=input_da_nan_mask,
    monthly_da=monthly_da,
    padded_days_mask=padded_days_mask,
    time_features=time_features,
    land_mask=lsm_subset["lsm"],
    patch_size=(1, *spatial_patch_size),  # based on the patch_size in model
    stride=stride,
    sh_embed_dim=96,
    sh_order_L = 10,
    verbose=True,
    load_lazy=True,
)
print(len(dataset_train))

#### Prepare validation data

In [ ]:
data_dir = f"{run_dir}/data_validation"
input_da, input_da_nan_mask, monthly_da, padded_days_mask, time_features = data_preparation(
    daily_validation_subset[var_name], monthly_validation_subset[var_name], calculate_residuals=True, is_hourly=False, save_to_zarr=True, run_dir=data_dir,
)

In [ ]:
# read data 
data_dir = f"{run_dir}/data_validation"
input_da, input_da_nan_mask, monthly_da, padded_days_mask, time_features = read_st_data(data_path=data_dir, var_name=var_name)

In [ ]:
num_patches = (10, 10)
spatial_patch_size = (patch_size[1]*num_patches[0], patch_size[2]*num_patches[1])
stride = (spatial_patch_size[0] // 5, spatial_patch_size[1] // 5)

dataset_validation = STDataset(
    input_da=input_da,
    input_da_nan_mask=input_da_nan_mask,
    monthly_da=monthly_da,
    padded_days_mask=padded_days_mask,
    time_features=time_features,
    land_mask=lsm_subset["lsm"],
    patch_size=(1, *spatial_patch_size),  # based on the patch_size in model
    stride=stride,
    sh_embed_dim=96,
    sh_order_L = 10,
    verbose=True,
    load_lazy=True,
)
print(len(dataset_validation))

In [ ]:
# create dataloader config
dataloader_config = DataLoaderConfig(
    batch_size=10,
    shuffle=True,
    num_workers= dataloader_num_workers,
    pin_memory=False,
    persistent_workers=True,
    device=device,
)
dataloader_config

In [ ]:
from torch.utils.data import DataLoader

dataloader = DataLoader(
    dataset_train,
    batch_size=dataloader_config.batch_size,
    shuffle=dataloader_config.shuffle,
    pin_memory=False,
    num_workers=dataloader_config.num_workers,  # for data loading
    persistent_workers=True,  # keep workers alive between epochs
)
num_batches = len(dataloader)
print(num_batches)

In [ ]:
%%time
batch = next(iter(dataloader))

In [ ]:
# create training config
training_config = TrainConfig(
    calculate_residuals=True,
    num_epoch=100,
    patience=10,
    accumulation_steps=2,
    optimizer_lr=1e-3,
    device=device,
    verbose=True,
    verbose_epoch_interval=20,
    tune_checkpoint=False,
    store_model=True,
)
training_config

### Start training loop

In [ ]:
%%time
# verbose is True
trained_model = train_monthly_model(
    model=model,
    dataset_train=dataset_train,
    dataloader_config=dataloader_config,
    training_config=training_config,
    dataset_validation=dataset_validation,
    run_dir=run_dir,
)

## Inspect results and compare

In [ ]:
plot_loss(run_dir, list_loss_var=["Loss/train", "Loss/validation"], unit="K")

#### Prepare the test data

In [ ]:
data_dir = f"{run_dir}/data_test"
input_da, input_da_nan_mask, monthly_da, padded_days_mask, time_features = data_preparation(
    daily_test_subset[var_name], monthly_test_subset[var_name], calculate_residuals=True, is_hourly=False, save_to_zarr=True, run_dir=data_dir,
)

In [ ]:
# read data 
data_dir = f"{run_dir}/data_test"
input_da, input_da_nan_mask, monthly_da, padded_days_mask, time_features = read_st_data(data_path=data_dir, var_name=var_name)

In [ ]:
num_patches = (10, 10)
spatial_patch_size = (patch_size[1]*num_patches[0], patch_size[2]*num_patches[1])
stride = (spatial_patch_size[0] // 5, spatial_patch_size[1] // 5)

dataset_test = STDataset(
    input_da=input_da,
    input_da_nan_mask=input_da_nan_mask,
    monthly_da=monthly_da,
    padded_days_mask=padded_days_mask,
    time_features=time_features,
    land_mask=lsm_subset["lsm"],
    patch_size=(1, *spatial_patch_size),  # based on the patch_size in model
    stride=stride,
    sh_embed_dim=96,
    sh_order_L = 10,
    verbose=True,
    load_lazy=False,
)
print(len(dataset_test))

In [ ]:
# create prediction config
prediction_config = PredictionConfig(
    calculate_residuals=True,
    return_numpy=True,
    save_predictions=True,
    return_loss=False,
    device=device,
    verbose=True,
)

In [ ]:
# inference on test data, verbose is True
predictions = predict_monthly_var(
    model=f"{run_dir}/best_model.pth",
    dataset=dataset_test,  
    dataloader_config=dataloader_config,
    prediction_config=prediction_config,
    run_dir=run_dir,
)

In [ ]:
predictions_res = xr.open_dataset(f"{run_dir}/202103_{var_name}_prediction_residual.nc")

In [ ]:
# add to monthly average
input_data_averaged = daily_test_subset.resample({"time": "MS"}).mean(skipna=True)
input_data_averaged["time"] = monthly_test_subset["time"]
predictions = input_data_averaged[var_name] + predictions_res[var_name]

In [ ]:
ocean = ~lsm_subset["lsm"].values
masked_target = monthly_test_subset[var_name].where(ocean)
masked_pred = predictions.where(ocean)
plot_results(masked_target, masked_pred, label="SST K", title=("Target", "Prediction"))

In [ ]:
err_baseline = (monthly_test_subset[var_name] - input_data_averaged[var_name])   # M - mean(D)
err_predictions = (monthly_test_subset[var_name] - predictions)  # M - M_hat

ocean = ~lsm_subset["lsm"].values
masked_err_baseline = err_baseline.where(ocean)
masked_err_predictions = err_predictions.where(ocean) 

plot_results(masked_err_baseline, masked_err_predictions, label="error K", title=("err_Baseline", "err_Prediction"), error=True)

In [ ]:
plot_histograms(masked_err_baseline, masked_err_predictions, label="error K", legend_labels=("err_Baseline", "err_Prediction"))